Use this to wipe the database before recreating:
```
PRAGMA writable_schema = 1;
delete from sqlite_master where type in ('table', 'index', 'trigger');
PRAGMA writable_schema = 0;
VACUUM;
PRAGMA INTEGRITY_CHECK;
```

In [1]:
import sqlite3
import pandas as pd
from io import StringIO
from pathlib import Path
from dbml_sqlite import toSQLite

In [3]:
dbml_filepath = '../diagram.dbml'
ddl_string = toSQLite(dbml_filepath)

In [4]:
print(ddl_string)

CREATE TABLE IF NOT EXISTS ORGANISM (
  TAXCODE INTEGER PRIMARY KEY,
  NAME TEXT
);
CREATE TABLE IF NOT EXISTS MODIFICATION (
  MODIFICATION_ID INTEGER PRIMARY KEY,
  NAME TEXT,
  SYMBOL TEXT,
  MODIFIABLE_RESIDUES TEXT
);
CREATE TABLE IF NOT EXISTS PROTEASE (
  PROTEASE_ID INTEGER PRIMARY KEY,
  NAME TEXT,
  CLEAVAGE_RULE TEXT
);
CREATE TABLE IF NOT EXISTS OMIC (
  OMIC_ID INTEGER PRIMARY KEY,
  NAME TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS ENRICHMENT_TYPE (
  ENRICHMENT_TYPE_ID INTEGER PRIMARY KEY,
  NAME TEXT NOT NULL,
  DESCRIPTION TEXT NOT NULL,
  REFERENCE TEXT,
  SHORT TEXT NOT NULL,
  ENRICHMENT_CLASS TEXT NOT NULL
);
CREATE TABLE IF NOT EXISTS ENRICHMENT_TYPE_TO_OMIC (
  ENRICHMENT_TYPE_ID INTEGER,
  OMIC_ID INTEGER,
  FOREIGN KEY(ENRICHMENT_TYPE_ID) REFERENCES ENRICHMENT_TYPE(ENRICHMENT_TYPE_ID),
  FOREIGN KEY(OMIC_ID) REFERENCES OMIC(OMIC_ID)
);
CREATE TABLE IF NOT EXISTS ENZYME_CLASS (
  ENZYME_CLASS_ID INTEGER PRIMARY KEY,
  NAME TEXT NOT NULL
);
CREATE TABLE IF NOT EXI

In [5]:
conn = sqlite3.connect('../sqlite_backend.db')
conn.executescript(ddl_string)

In [19]:
organism_df = pd.DataFrame({
    'TAXCODE': [9606, 10090],
    'NAME': ['Homo sapiens', 'Mus musculus']})
organism_df

,TAXCODE,NAME
0,9606,Homo sapiens
1,10090,Mus musculus


In [20]:
organism_df.to_sql('ORGANISM', conn, if_exists='append', index=False)

2

In [23]:
modification_df = pd.DataFrame({
    'MODIFICATION_ID': [1],
    'NAME': ['Phosphorylation'],
    'SYMBOL': ['(ph)'],
    'MODIFIABLE_RESIDUES': ['STY']
})
modification_df

,MODIFICATION_ID,NAME,SYMBOL,MODIFIABLE_RESIDUES
0,1,Phosphorylation,(ph),STY


In [24]:
modification_df.to_sql('MODIFICATION', conn, if_exists='append', index=False)

1

In [25]:
protease_df = pd.DataFrame({
    'PROTEASE_ID':[1],
    'NAME':['Trypsin'],
    'CLEAVAGE_RULE':['([KR])(?!P)']

})
protease_df

,PROTEASE_ID,NAME,CLEAVAGE_RULE
0,1,Trypsin,([KR])(?!P)


In [26]:
protease_df.to_sql('PROTEASE', conn, if_exists='append', index=False)

1

In [27]:
omic_df = pd.DataFrame({
    'OMIC_ID':[1,2,3],
    'NAME':['Phosphorylation','Protein','Others']
})
omic_df

,OMIC_ID,NAME
0,1,Phosphorylation
1,2,Protein
2,3,Others


In [28]:
omic_df.to_sql('OMIC', conn, if_exists='append', index=False)

3

In [29]:
enrichment_type_df = pd.read_csv(StringIO("""
ENRICHMENT_TYPE_ID,NAME,DESCRIPTION,REFERENCE,SHORT,ENRICHMENT_CLASS
1,PTM-SEA,PTM-Centric Enrichment Analysis using the PTM Signature Database (PTMSigDB),https://github.com/broadinstitute/ssGSEA2.0,ptmsea,Pathway
2,GC-PEA,Gene-Centric Pathway Enrichment Analysis. Basically a GSEA against a database of pathway signatures,https://github.com/broadinstitute/ssGSEA2.0,gc,Pathway
3,GCR-PEA,"Gene-Centric Redundant Pathway Enrichment Analysis. Basically a GSEA against a database of pathway signatures, but genes are counted repeatedly for each regulated site in the data.",https://github.com/broadinstitute/ssGSEA2.0,gcr,Pathway
4,KSEA,KSEA uses phosphoproteomics data (usually fold changes) and prior knowledge on kinase-substrate relationships to infer kinase activities.,https://github.com/saezlab/kinact,ksea,KinaseActivity
5,RoKAI+KSEA,RoKAI refines phosphorylation profiles and has been shown to produce more robust results when combined with any kinase activity inference method,https://github.com/serhan-yilmaz/RokaiApp,ksea_rokai,KinaseActivity
6,MOTIF,Kinase Motif Enrichment Analysis. Uses the Kinase Library published by Johnson et al. Code reverse-engineered by Florian Bayer.,,motif,KinaseActivity
7,KEA3,Kinase Enrichment Analysis 3 (KEA3) infers upstream kinases whose putative substrates are overrepresented in a user-inputted list of proteins or differentially phosphorylated proteins.,https://maayanlab.cloud/kea3/templates/api.jsp,kea3,KinaseActivity
8,KSTAR,Kinase-Substrate Translation to Activity Relationships. A Kinase Activity Enrichment algorithm.,https://github.com/NaegleLab/KSTAR,kstar,KinaseActivity
9,RoKAI,"RoKAI refinement, followed by RoKAIs own kinase activity inference method.",https://github.com/serhan-yilmaz/RokaiApp,rokai,KinaseActivity
10,GO,Gene Ontology Enrichment using Fishers exact test,"https://www.nature.com/articles/ng0500_25, https://academic.oup.com/genetics/article/224/1/iyad031/7068118",go,Pathway
"""))
enrichment_type_df

,ENRICHMENT_TYPE_ID,NAME,DESCRIPTION,REFERENCE,SHORT,ENRICHMENT_CLASS
0,1,PTM-SEA,PTM-Centric Enrichment Analysis using the PTM ...,https://github.com/broadinstitute/ssGSEA2.0,ptmsea,Pathway
1,2,GC-PEA,Gene-Centric Pathway Enrichment Analysis. Basi...,https://github.com/broadinstitute/ssGSEA2.0,gc,Pathway
2,3,GCR-PEA,Gene-Centric Redundant Pathway Enrichment Anal...,https://github.com/broadinstitute/ssGSEA2.0,gcr,Pathway
3,4,KSEA,KSEA uses phosphoproteomics data (usually fold...,https://github.com/saezlab/kinact,ksea,KinaseActivity
4,5,RoKAI+KSEA,RoKAI refines phosphorylation profiles and has...,https://github.com/serhan-yilmaz/RokaiApp,ksea_rokai,KinaseActivity
5,6,MOTIF,Kinase Motif Enrichment Analysis. Uses the Kin...,NaN,motif,KinaseActivity
6,7,KEA3,Kinase Enrichment Analysis 3 (KEA3) infers ups...,https://maayanlab.cloud/kea3/templates/api.jsp,kea3,KinaseActivity
7,8,KSTAR,Kinase-Substrate Translation to Activity Relat...,https://github.com/NaegleLab/KSTAR,kstar,KinaseActivity
8,9,RoKAI,"RoKAI refinement, followed by RoKAIs own kinas...",https://github.com/serhan-yilmaz/RokaiApp,rokai,KinaseActivity
9,10,GO,Gene Ontology Enrichment using Fishers exact test,"https://www.nature.com/articles/ng0500_25, htt...",go,Pathway


In [31]:
enrichment_type_df.to_sql('ENRICHMENT_TYPE', conn, if_exists='append', index=False)

10

In [32]:
enrichment_type_to_omic_df = pd.DataFrame({
    'ENRICHMENT_TYPE_ID':[1,2,2,2,3,3,3,4,5,6,7,8,9,10,10,10],
    'OMIC_ID':[1,1,2,3,1,2,3,1,1,1,1,1,1,1,2,3],
})
enrichment_type_to_omic_df

,ENRICHMENT_TYPE_ID,OMIC_ID
0,1,1
1,2,1
2,2,2
3,2,3
4,3,1
5,3,2
6,3,3
7,4,1
8,5,1
9,6,1


In [33]:
enrichment_type_to_omic_df.to_sql('ENRICHMENT_TYPE_TO_OMIC', conn, if_exists='append', index=False)

16

In [34]:
enzyme_class_df = pd.DataFrame({
    'ENZYME_CLASS_ID':[1],
    'NAME':['Kinase'],
})
enzyme_class_df

,ENZYME_CLASS_ID,NAME
0,1,Kinase


In [35]:
enzyme_class_df.to_sql('ENZYME_CLASS', conn, if_exists='append', index=False)

1

In [36]:
conn.close()